[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CONNECTS-SCV/bio-guides/blob/main/humanization/humanization_guide/06_cdr_safe_tools/06_tools_lab.ipynb)


# 06 — CDR-safe 도구 — Humatch · AnthroAb · 3도구 합의

> 본문 [`06_cdr_safe_tools.md`](06_cdr_safe_tools.md) 와 **한 절씩 짝지어** 보세요.
> **① 직접 실행 → ② 내 결과 확인 → ③ 레퍼런스 대조** 순서로 진행해요.
> 이 노트북의 표·그래프·수치는 **여러분이 방금 돌린 `my_run/`** 에서 나와요. 실행을 건너뛰거나 실패하면 저장소에 커밋된 `data/`(실제 실행 산출물) 로 자동 폴백해요.
>
> **실측 소요 —** Humatch 첫 실행 **160초**(Zenodo CNN 가중치 다운로드 포함) · AnthroAb `predict_best_score` **1.3~1.5초** / masked **2.3~2.5초**

## 0) Colab 준비 — 저장소 클론 & 작업 폴더 이동

- **Colab**: 이 셀을 그대로 실행하세요. 저장소를 클론하고 이 챕터(`06_cdr_safe_tools`) 폴더로 이동한 뒤, 필요한 패키지만 깝니다.
- **로컬**: 이 노트북을 `06_cdr_safe_tools/` 폴더 안에서 열었다면 클론 없이 그대로 진행돼요.

이 셀이 끝나면 두 개의 경로가 준비돼요 — **`my_run/`**(내가 직접 만들 결과)과 **`data/`**(저장소에 커밋된 레퍼런스 결과).
아래 랩은 항상 `my_run/` 을 먼저 찾고, 없으면 `data/` 로 폴백하면서 **어느 쪽을 쓰는지 print** 합니다.
- ANARCI 는 numbering 을 **hmmscan(HMMER)** 서브프로세스로 돌려요. Colab 에서는 `apt-get install -y hmmer` 가 함께 실행돼요 — pip 만으로는 `hmmscan` 이 없어 죽어요.

In [ ]:
# ====== Colab/로컬 공용 부트스트랩 (모든 챕터 공통) ======
REPO_URL = "https://github.com/CONNECTS-SCV/bio-guides.git"   # 이 가이드 저장소 (fork 했다면 본인 주소로 바꾸세요)
CLONE_AS = "bio-guides"
CHAPTER  = "06_cdr_safe_tools"
APT_PKGS = "hmmer"     # Colab 에서만: 시스템 패키지 (hmmer = ANARCI 가 부르는 hmmscan)
PIP_PKGS = "pandas matplotlib anarci abnumber"     # 없는 것만 설치

import os, sys, json, time, shutil, pathlib, subprocess, importlib, importlib.util
IN_COLAB = "google.colab" in sys.modules

# HuggingFace 가중치 다운로드가 '멈춘 채' 매달리는 일을 막아요.
# (멈춤은 예외가 안 나서 폴백이 안 걸려요 — 타임아웃을 걸어 실패로 바꿔야 data/ 로 이어져요)
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "30")   # 스트림 30초 무응답 → 끊고 재시도
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "15")

def _run(cmd, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, check=check)

_MARK = "humanization_viz.py"          # 이 파일이 있는 폴더 = 가이드 루트

def _find_root():
    cwd = pathlib.Path.cwd()
    for base in (cwd, *list(cwd.parents)[:3]):
        if (base / _MARK).exists():
            return base
    # 클론 직후: cwd '아래'만 깊이 3까지 훑는다.
    # (상위로 올라가 rglob 하면 Colab 에서는 / 전체를 뒤지게 된다 — 매우 느리고 엉뚱한 경로를 잡는다)
    for pat in (f"*/{_MARK}", f"*/*/{_MARK}", f"*/*/*/{_MARK}"):
        hits = sorted(cwd.glob(pat))
        if hits:
            return hits[0].parent
    return None

ROOT = _find_root()
if ROOT is None and IN_COLAB:
    if not pathlib.Path(CLONE_AS).exists():
        _run(f'git clone --depth 1 "{REPO_URL}" {CLONE_AS}')
    ROOT = _find_root()
assert ROOT is not None, "가이드 루트를 못 찾았어요. 로컬이면 이 노트북을 챕터 폴더 안에서 여세요."

GUIDE_ROOT = ROOT.resolve()
os.chdir(GUIDE_ROOT / CHAPTER)         # data/ 상대경로가 그대로 동작
sys.path.insert(0, str(GUIDE_ROOT))    # humanization_viz import
sys.path.insert(0, str(GUIDE_ROOT / CHAPTER))

_IMPORT_NAME = {"biopython": "Bio", "pyyaml": "yaml", "scikit-learn": "sklearn"}

def _have(mod):
    try:
        return importlib.util.find_spec(mod) is not None
    except Exception:
        return False

def _ensure(pkgs):
    miss = [p for p in pkgs.split() if not _have(_IMPORT_NAME.get(p, p.replace("-", "_")))]
    if miss:
        print("설치:", " ".join(miss))
        _run(f'"{sys.executable}" -m pip -q install ' + " ".join(miss))
        importlib.invalidate_caches()

# ANARCI 는 numbering 을 hmmscan(HMMER) 서브프로세스로 돌려요 — pip 만으로는 hmmscan 이 없어요.
if APT_PKGS and IN_COLAB:
    _run("apt-get -qq update")                 # 인덱스가 낡으면 install 이 404 로 죽는다
    _run(f"apt-get -qq install -y {APT_PKGS}")
if PIP_PKGS:
    _ensure(PIP_PKGS)

def _ensure_pkg_resources():
    # setuptools 81+(2026-02) 이 pkg_resources 를 없앴는데 IgFold 의존성이 이걸 import 합니다.
    if importlib.util.find_spec("pkg_resources") is None:
        _run(f'"{sys.executable}" -m pip -q install "setuptools<81"')
        importlib.invalidate_caches()

import glob as _glob
if IN_COLAB and not _glob.glob("/usr/share/fonts/**/*Nanum*", recursive=True):
    # Colab 에는 한글 폰트가 없어 그래프의 한글 라벨이 □ 로 깨져요.
    _run("apt-get -qq update"); _run("apt-get -qq install -y fonts-nanum")


# ── ① 내가 만든 결과(my_run) 우선 · ② 없으면 커밋된 레퍼런스(data/) ─────────────
MY  = pathlib.Path("my_run"); MY.mkdir(exist_ok=True)
REF = pathlib.Path("data")

def find_one(pattern, ref=REF, quiet=False):
    """산출물 하나 찾기 — 내가 만든 my_run/ 을 먼저 뒤지고, 없으면 커밋된 레퍼런스에서."""
    hits = sorted(MY.rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과]   {hits[0]}")
        return hits[0]
    hits = sorted(pathlib.Path(ref).glob(pattern))
    assert hits, f"'{pattern}' 을 my_run/ 에서도 {ref}/ 에서도 찾지 못했어요."
    if not quiet: print(f"[레퍼런스] {hits[0]}")
    return hits[0]

def find_prev(chapter, pattern, ref=REF, quiet=False):
    """앞 챕터에서 직접 만든 산출물 → 없으면 이 챕터 data/ 레퍼런스."""
    hits = sorted((GUIDE_ROOT / chapter / "my_run").rglob(pattern))
    if hits:
        if not quiet: print(f"[내 결과 · {chapter}] {hits[0]}")
        return hits[0]
    return find_one(pattern, ref, quiet)

def read_fasta(path):
    out, name, buf = {}, None, []
    for line in pathlib.Path(path).read_text().splitlines():
        if line.startswith(">"):
            if name: out[name] = "".join(buf)
            name, buf = line[1:].strip(), []
        elif line.strip():
            buf.append(line.strip())
    if name: out[name] = "".join(buf)
    return out

def write_fasta(path, records):
    pathlib.Path(path).write_text("".join(f">{k}\n{v}\n" for k, v in records.items()))
    return pathlib.Path(path)

PARENTAL = read_fasta(REF / "parental.fasta")      # 가이드 전체를 관통하는 러닝 예제
VH = PARENTAL["parental_H"]
VL = PARENTAL["parental_L"]

print("가이드 루트 :", GUIDE_ROOT)
print("작업 폴더   :", pathlib.Path.cwd())
print(f"러닝 예제   : VH {len(VH)} aa · VL {len(VL)} aa (mouse hybridoma 가정)")

## 1) 직접 실행 — Humatch 설치 + humanization (본문 6.1)

Humatch 는 **CDR 보호가 도구에 내장**돼 있어요(`allow_CDR_mutations=False` 가 기본값).
대신 framework 를 CNN 점수 목표(0.95)에 닿을 때까지 single-point 로 반복 탐색해요.

첫 실행은 **Zenodo 에서 CNN 가중치(heavy/light/paired) + germline 룩업 배열**을 받으므로 **160초** 걸려요(실측).
`pip install humatch` 가 안 되면 **GitHub source** 로 자동 전환해요.

In [ ]:
import pandas as pd

# Humatch 는 파이썬 모듈명이 `Humatch`(대문자) 라서 소문자 import 체크가 통하지 않아요 → CLI 존재로 판정
if not shutil.which("Humatch-humanise"):
    _run(f'"{sys.executable}" -m pip -q install humatch', check=False)
if not shutil.which("Humatch-humanise"):
    print("pip 로 안 되면 GitHub source 로 (본문 6.1.1 케이스 스터디)")
    _run(f'"{sys.executable}" -m pip -q install git+https://github.com/oxpig/Humatch.git', check=False)
print("Humatch CLI:", shutil.which("Humatch-humanise") or "없음")

# CLI 는 CSV 입력/CSV 출력이 가장 안전해요(문자열 인자 -H/-L 도 가능).
inp = MY / "humatch_in.csv"
inp.write_text("VH,VL\n%s,%s\n" % (VH, VL))
out = MY / "humatch_out.csv"
cmd = ["Humatch-humanise", "-i", str(inp), "--vh_col", "VH", "--vl_col", "VL", "-o", str(out), "-v"]

hm_h = hm_l = None
env = dict(os.environ, TF_CPP_MIN_LOG_LEVEL="3")
t0 = time.time()
try:
    r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if r.returncode != 0 and "DNN library" in (r.stdout + r.stderr):
        # TensorFlow 가 GPU cuDNN 초기화에 실패하는 환경이 있어요 → CPU 로 강제하고 재시도(실측 사례)
        print("TensorFlow GPU 초기화 실패 → CPU 로 재시도해요")
        env["CUDA_VISIBLE_DEVICES"] = ""
        r = subprocess.run(cmd, capture_output=True, text=True, env=env)
    if r.returncode == 0 and out.exists():
        print(f"Humatch 완료: {time.time()-t0:.1f}초 (첫 실행은 Zenodo 가중치 다운로드 포함 — 실측 160초)")
        hm = pd.read_csv(out).iloc[0]
        hm_h = hm["Humatch_H"].replace("-", "")     # 출력은 200-position 정렬형 → gap 제거
        hm_l = hm["Humatch_L"].replace("-", "")
        write_fasta(MY / "humatch_humanised.fasta",
                    {"humatch_humanised_VH": hm_h, "humatch_humanised_VL": hm_l})
        print(f"CNN_H={hm['CNN_H']:.3f}  CNN_L={hm['CNN_L']:.3f}  CNN_P={hm['CNN_P']:.3f}  "
              f"HV={hm['HV']}  LV={hm['LV']}  edit={hm['Edit']}")
    else:
        print("Humatch 실행 실패:", (r.stdout + r.stderr).strip()[-300:])
except Exception as e:
    print("Humatch 실행 실패:", type(e).__name__, str(e)[:200])

if hm_h is None:
    print("→ 레퍼런스(data/humatch_humanized.fasta)로 이어가요.")
    ref = read_fasta("data/humatch_humanized.fasta")
    hm_h, hm_l = ref["humatch_humanised_VH"], ref["humatch_humanised_VL"]

## 2) 내 결과 확인 — Humatch 는 CDR 을 정말 안 건드렸나

Humatch 는 VL 에서 **1 잔기를 지워요**(우리 예제: 111 → 110 aa). 그래서 raw 인덱스 비교가 깨져요 —
**CDR 보존 확인은 "parental CDR 문자열이 후보 안에 그대로 있는가"** 로 하는 게 안전해요.

In [ ]:
ct = pd.read_csv(find_one("cdr_table_imgt.csv", quiet=True))
cdrs = {(r["chain"], r["cdr"]): r["sequence"] for _, r in ct.iterrows()}

def cdr_intact(cand_h, cand_l, label):
    rows = []
    for (chain, name), s in cdrs.items():
        seq = cand_h if chain == "H" else cand_l
        rows.append({"후보": label, "CDR": f"{chain}-{name}", "parental CDR": s, "보존": s in seq})
    return rows

chk = pd.DataFrame(cdr_intact(hm_h, hm_l, "Humatch"))
display(chk)
print("VL 길이 — parental", len(VL), "→ Humatch", len(hm_l), "(1 잔기 삭제 = indel)")
print("→ CDR 보존:", int(chk["보존"].sum()), "/ 6.  Humatch 는 CDR 을 안 건드려요(도구에 내장된 보호).")

## 3) 직접 실행 — AnthroAb 두 모드 (본문 6.2.1)

| 모드 | 함수 | 무엇을 바꾸나 |
|---|---|---|
| ① 자동 전체 변경 | `predict_best_score(seq, chain)` | **모든 position** 을 가장 사람다운 잔기로 — **CDR 도 바꿔요** |
| ② 커스텀 마스킹 | `predict_masked(seq, chain)` | 내가 `*` 로 찍은 자리만 |

먼저 ① 을 그대로 돌려요(실측 VH 1.5초 / VL 1.3초).

In [ ]:
_ensure("anthroab")

def mutations(par, hum):
    return [{"position_1based": i + 1, "wt": a, "mut": b, "mutation": f"{a}{i+1}{b}"}
            for i, (a, b) in enumerate(zip(par, hum)) if a != b]

ab_h = ab_l = None
try:
    import anthroab
    t0 = time.time()
    ab_h = anthroab.predict_best_score(VH, "H")
    ab_l = anthroab.predict_best_score(VL, "L")
    print(f"predict_best_score VH+VL: {time.time()-t0:.1f}초  (실측 1.3~1.5초/체인)")
    write_fasta(MY / "anthroab_best_score.fasta",
                {"anthroab_bestscore_VH": ab_h, "anthroab_bestscore_VL": ab_l})
except Exception as e:
    print("AnthroAb 실행 실패:", type(e).__name__, str(e)[:160])

if ab_h is None:
    print("[레퍼런스] data/anthroab_best_score.fasta")
    f = read_fasta("data/anthroab_best_score.fasta")
    ab_h, ab_l = f["anthroab_predict_best_score_VH"], f["anthroab_predict_best_score_VL"]

ab_mut_h = pd.DataFrame(mutations(VH, ab_h)); ab_mut_l = pd.DataFrame(mutations(VL, ab_l))
print(f"VH {len(ab_mut_h)} mut · VL {len(ab_mut_l)} mut")

chk2 = pd.DataFrame(cdr_intact(ab_h, ab_l, "AnthroAb(best_score)"))
display(chk2)
print("→ CDR 보존:", int(chk2["보존"].sum()), "/ 6.  자동 모드는 CDR 을 지켜주지 않아요(본문 6.2.1 경고 그대로).")

## 4) 직접 실행 — `predict_masked` 의 버그와 우회 (실측 발견)

**anthroab 1.1.0 의 `predict_masked()` 는 그대로 쓰면 안 됩니다.** docstring 은 `*`/`X` 로 마스킹하라고 하지만,
`hemantn/roberta-base-humAb-*` 의 tokenizer vocab 에는 `*`·`X` 가 **없어요**. 사전에 없는 문자는 `<unk>` 도 아니고 **조용히 삭제**되어,
그 뒤 토큰이 한 칸씩 밀려요. 라이브러리는 (원래 길이의) 입력과 (짧아진) 예측을 zip 해서 **말없이 어긋난 결과**를 내거나 `IndexError` 로 죽어요.

우회는 간단해요 — **문자열 마스킹을 건너뛰고 `input_ids` 를 직접 만들어** 진짜 `mask_token_id` 를 꽂아요.
여기서는 **FR 자리만** 마스킹해(=CDR 은 구조적으로 보존) 돌려요.

In [ ]:
_ensure("torch transformers")
import torch
from transformers import AutoModelForMaskedLM, AutoTokenizer

MODELS = {"H": "hemantn/roberta-base-humAb-vh", "L": "hemantn/roberta-base-humAb-vl"}

@torch.no_grad()
def masked_fill(seq, chain, mask_positions):
    """mask_positions(1-based)만 마스킹해 예측 — tokenizer 문자열 마스킹을 우회한 정공법."""
    tok = AutoTokenizer.from_pretrained(MODELS[chain])
    mdl = AutoModelForMaskedLM.from_pretrained(MODELS[chain]); mdl.eval()
    vocab = tok.get_vocab()
    ids = [tok.bos_token_id] + [vocab[a] for a in seq] + [tok.eos_token_id]
    x = torch.tensor(ids).unsqueeze(0)
    for p in mask_positions:
        x[0, p] = tok.mask_token_id                 # +1 offset (bos) 이 이미 반영된 인덱스
    logits = mdl(input_ids=x).logits[0]
    inv = {v: k for k, v in vocab.items()}
    out = list(seq)
    for p in mask_positions:
        out[p - 1] = inv[int(logits[p].argmax())]
    return "".join(out)

# CDR 보호 좌표(Ch.04) → FR 자리만 마스킹
gp = GUIDE_ROOT / "04_sequence_qc" / "my_run" / "cdr_guard.json"
if gp.exists():
    guard = json.loads(gp.read_text()); print(f"[내 결과 · 04_sequence_qc] {gp}")
else:
    guard = {"H": [], "L": []}
    for _, r in ct.iterrows():
        seq = VH if r["chain"] == "H" else VL
        st = seq.find(r["sequence"]) + 1
        guard[r["chain"]] += list(range(st, st + len(r["sequence"])))
    print("[레퍼런스] data/cdr_table_imgt.csv 로 CDR 좌표 복원")

fr_h = [i for i in range(1, len(VH) + 1) if i not in guard["H"]]
fr_l = [i for i in range(1, len(VL) + 1) if i not in guard["L"]]

mk_h = mk_l = None
try:
    t0 = time.time()
    mk_h = masked_fill(VH, "H", fr_h)
    mk_l = masked_fill(VL, "L", fr_l)
    print(f"predict_masked(우회 구현) VH+VL: {time.time()-t0:.1f}초  — 실측 2.3~2.5초/체인")
    write_fasta(MY / "anthroab_masked_FRonly.fasta",
                {"anthroab_masked_VH": mk_h, "anthroab_masked_VL": mk_l})
except Exception as e:
    print("masked 실행 실패:", type(e).__name__, str(e)[:160])
    print("[레퍼런스] data/anthroab_masked_FRonly.fasta")
    f = read_fasta("data/anthroab_masked_FRonly.fasta")
    mk_h, mk_l = f["anthroab_predict_masked_fixed_VH"], f["anthroab_predict_masked_fixed_VL"]

mk_mut_h = pd.DataFrame(mutations(VH, mk_h)); mk_mut_l = pd.DataFrame(mutations(VL, mk_l))
print(f"FR-only masked → VH {len(mk_mut_h)} mut · VL {len(mk_mut_l)} mut")
chk3 = pd.DataFrame(cdr_intact(mk_h, mk_l, "AnthroAb(masked FR-only)"))
print("CDR 보존:", int(chk3["보존"].sum()), "/ 6 (구조적으로 보장 — CDR 을 아예 마스킹하지 않았으니까)")

## 5) 직접 실행 — 3도구 합의 분석 (본문 6.2.6)

여기가 이 챕터의 핵심이에요. **도구 간 비교는 반드시 IMGT 번호로** 합니다(Humatch 의 indel 때문에 raw 인덱스가 어긋나므로).
Ch.04 에서 만든 `raw2imgt_*.json` 을 그대로 씁니다.

In [ ]:
# raw → IMGT 크로스워크
def load_map(tag):
    p = GUIDE_ROOT / "04_sequence_qc" / "my_run" / f"raw2imgt_{tag}.json"
    if p.exists():
        print(f"[내 결과 · 04_sequence_qc] {p}")
    else:
        p = pathlib.Path(f"data/raw2imgt_{tag}.json"); print(f"[레퍼런스] {p}")
    return {int(k): v for k, v in json.loads(p.read_text()).items()}

r2i = {"H": load_map("H"), "L": load_map("L")}

# 각 도구의 mutation → IMGT 라벨 (Sapiens 는 Ch.05 my_run, 없으면 data/)
sap_h = pd.read_csv(find_prev("05_humanize_sapiens", "mutations_VH.csv"))
sap_l = pd.read_csv(find_prev("05_humanize_sapiens", "mutations_VL.csv"))

def to_imgt(df, chain, tool):
    return pd.DataFrame([{"chain": chain, "imgt": r2i[chain][int(r.position_1based)],
                          "wt": r.wt, "mut": r.mut, "sub": f"{r.wt}{r.mut}", "tool": tool}
                         for r in df.itertuples()])

tbl = pd.concat([
    to_imgt(sap_h, "H", "Sapiens"), to_imgt(sap_l, "L", "Sapiens"),
    to_imgt(ab_mut_h, "H", "AnthroAb_best_score"), to_imgt(ab_mut_l, "L", "AnthroAb_best_score"),
    to_imgt(mk_mut_h, "H", "AnthroAb_masked"), to_imgt(mk_mut_l, "L", "AnthroAb_masked"),
], ignore_index=True)

# Humatch 는 이미 IMGT 로 나온 레퍼런스 표를 쓰거나, 내 결과에서 IMGT 로 재계산
hmt = pd.read_csv("data/humatch_mutations_imgt.csv")
hmt = pd.DataFrame([{"chain": r["chain"], "imgt": r["imgt_position"], "wt": r["wt"], "mut": r["mut"],
                     "sub": f"{r['wt']}{r['mut']}", "tool": "Humatch"} for _, r in hmt.iterrows()])
tbl = pd.concat([tbl, hmt], ignore_index=True)
tbl.to_csv(MY / "mutations_by_tool_imgt.csv", index=False)

def consensus(anthroab_mode):
    keep = tbl[tbl.tool.isin(["Sapiens", "Humatch", anthroab_mode])]
    g = keep.groupby(["chain", "imgt"])
    out = []
    for (chain, pos), sub in g:
        tools = set(sub.tool)
        if len(tools) == 3:
            same = len(set(sub["sub"])) == 1
            out.append({"chain": chain, "imgt": pos,
                        "subs": "/".join(sorted(set(sub["sub"]))), "동일 치환": same})
    return pd.DataFrame(out).sort_values(["chain", "imgt"])

for mode in ["AnthroAb_best_score", "AnthroAb_masked"]:
    cs = consensus(mode)
    ident = cs[cs["동일 치환"]]
    print(f"\n=== 3도구(Sapiens·Humatch·{mode}) 합의 ===")
    print(f"세 도구가 모두 건드린 위치: {len(cs)}개 · 그중 **같은 잔기**로 바꾼 위치: {len(ident)}개")
    display(ident)
    cs.to_csv(MY / f"three_way_consensus_{mode}.csv", index=False)

## 6) 내 결과 확인 — 본문의 "78번이 유일한 3도구 합의" 는 사실이 아니에요

실측하면 이래요.

- **`I78T`(raw) = IMGT `H86`** 은 **진짜 3도구 합의**가 맞아요 — Sapiens·Humatch·AnthroAb(masked) 셋 다 `I→T`.
- 하지만 **"유일한" 합의는 아니에요.** AnthroAb 를 어느 모드로 비교하냐에 따라 **7개(best_score)** 또는 **12개(masked)** 의 3도구 합의 위치가 나와요.
- 게다가 `best_score` 모드의 AnthroAb 는 **raw 78 / IMGT H86 을 아예 건드리지 않아요** — 즉 그 비교에서는 `I78T` 가 합의 목록에 없어요.

교훈은 그대로 유효해요 — **여러 도구가 같은 자리에 같은 잔기를 제안하면 신뢰도가 올라간다.** 다만 "몇 개가 합의인가"는 **어느 모드로 비교했는지에 달렸어요.**

In [ ]:
from humanization_viz import mutation_map

# VH 만 · masked 모드 기준으로 위치별 도구 제안을 겹쳐 그려요(IMGT 번호축)
vh = tbl[(tbl.chain == "H") & (tbl.tool.isin(["Sapiens", "Humatch", "AnthroAb_masked"]))].copy()
vh["position"] = vh["imgt"].str.lstrip("H").astype(int)
shared = (vh.groupby("position")["tool"].nunique() == 3)
pos3 = sorted(shared[shared].index)                      # 세 도구가 모두 건드린 위치
rows = [{"position": r.position, "tool": r.tool, "to": r.mut}
        for r in vh.itertuples() if r.position in pos3]

mutation_map(rows, "3-tool proposals on VH (IMGT) — 금색=세 도구 동일 치환",
             "06_mutation_map.png", cdr_spans=[(27, 38), (56, 65), (105, 117)])
h86 = vh[vh.position == 86][["tool", "wt", "mut"]]
print("IMGT H86 (= raw I78T) 에 대한 세 도구의 제안:")
display(h86)
from IPython.display import Image; Image("06_mutation_map.png")

## 7) 레퍼런스 대조

In [ ]:
ref_cs = pd.read_csv("data/three_way_consensus.csv")
ref_ov = pd.read_csv("data/overlap_summary.csv")

print("[레퍼런스] 3도구 합의 — AnthroAb 모드별")
for mode, sub in ref_cs.groupby("anthroab_mode"):
    ident = sub[sub.identical_substitution_all3]
    print(f"  {mode:24s}: 세 도구 겹친 위치 {len(sub)}개 · 동일 치환 {len(ident)}개")
display(ref_cs[ref_cs.identical_substitution_all3][["anthroab_mode", "chain", "imgt_position",
                                                    "sapiens_mut", "humatch_mut", "anthroab_mut"]])

print("\n[레퍼런스] Sapiens × AnthroAb 겹침 — 본문의 '10개' 는 VL 만·best_score 모드일 때의 값이에요")
display(ref_ov[["comparison", "chain_scope", "n_positions", "note"]])

## 이 랩에서 확인한 것

1. **Humatch** — CNN_H **0.972** · CNN_L **1.000** · gene **hv1/lv2**, edit **20**(VH 18 · VL 2), **CDR mutation 0개**. CDR 보호가 도구에 내장돼 있어요. VL 은 1 잔기 **삭제**(indel) → 도구 간 비교는 IMGT 로.
2. **AnthroAb** — `predict_best_score` 는 CDR 을 **안 지켜요**(6개 CDR 중 일부 파손). `predict_masked` 는 **1.1.0 에서 깨져 있어**(vocab 에 `*` 없음 → 토큰 밀림) `input_ids` 를 직접 만들어 마스킹해야 합니다.
3. **3도구 합의(실측)** — `I78T`(IMGT `H86`) 는 진짜 합의지만 **유일하지 않아요.** 동일 치환 합의는 AnthroAb `best_score` 기준 **7개**, `masked` 기준 **12개**(`H5 Q→V`, `H21 M→V`, `H42 G→V`, `H74 A→G`, `H75 K→R`, `L99 G→E` 등). `best_score` 비교에서는 H86 이 합의에 **들어가지도 않아요**.
4. 본문의 "Sapiens × AnthroAb 겹침 10개" 는 **VL 만·best_score 모드**일 때의 숫자예요(VH 는 12개, VH+VL 은 22개).

다음 → **[Ch.07 — Nativeness](../07_nativeness/07_nativeness_lab.ipynb)**